# Tarifación paramétrica climática bajo no estacionariedad

**Notebook de investigación reproducible. DGP sintético: no es cotización ni evidencia sobre Colombia.** Compara burn analysis, ponderación temporal y precio condicional; mide riesgo de base, cola y asequibilidad.

## 1. Pregunta e hipótesis

H1: bajo deriva, una distribución condicional reduce error OOS de prima pura frente a burn estacionario; H2: optimizar trigger/agotamiento contra pérdida reduce shortfall fuera de muestra si la relación índice-daño persiste; H3: carga de cola aumenta suficiencia y puede perjudicar asequibilidad. Contraejemplo obligatorio: en clima estacionario la complejidad puede empeorar.

## 2. Contrato y fórmulas

Para índice de lluvia acumulada W (mm), trigger K, agotamiento A<K y límite L: \(q=\min(1,\max(0,(K-W)/(K-A)))\), \(Z=Lq\). Pérdida económica Y; base residual B=Y-Z; shortfall B+=max(B,0), pago excedente B-=max(-B,0). Prima pura \(E[Z]\); carga por cola \(\lambda(TVaR_\alpha(Z)-E[Z])\). Gastos, reaseguro y capital se informan por separado para evitar doble conteo.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
SEED=20261029; rng=np.random.default_rng(SEED); out=Path('results'); out.mkdir(exist_ok=True)
plt.rcParams.update({'figure.figsize':(9,4.5),'axes.grid':True})
print('seed',SEED,'synthetic experiment')

## 3. DGP transparente

La precipitación cambia de media y escala, con salto de régimen y tendencia. Daño económico aumenta con déficit de lluvia y tiene heterogeneidad idiosincrática. Los parámetros solo sirven para probar comportamiento del método.

In [ ]:
def simulate(n=1200, drift=True, seed=SEED):
 r=np.random.default_rng(seed); t=np.arange(n); regime=(t>=int(.58*n)).astype(float)
 mu=700-(.10*t+90*regime if drift else 0); sig=75+(.025*t if drift else 0)
 rain=np.maximum(0,mu+sig*r.standard_normal(n)); deficit=np.maximum(0,560-rain)
 vulnerability=r.lognormal(0,.25,n); loss=(.12*deficit+.0017*deficit**2)*vulnerability
 loss+=r.lognormal(1.4,.55,n)*(deficit>0)
 return pd.DataFrame({'t':t,'regime':regime,'mu':mu,'sigma':sig,'rain':rain,'loss':loss})
df=simulate(); df.describe().round(2)

## 4. Inspección del cambio

La distribución conjunta importa: histogramas agregados esconden tendencia y salto. En datos reales hay que auditar estación, instrumento, faltantes, ubicación, exposición y sesgo de medición.

In [ ]:
fig,ax=plt.subplots(); ax.plot(df.t,df.rain,lw=.7,alpha=.7); ax.plot(df.t,df.mu,lw=2,label='media DGP'); ax.axvline(int(.58*len(df)),ls='--',c='black',label='ruptura'); ax.axhline(560,ls=':',c='red',label='referencia'); ax.legend(); ax.set(xlabel='periodo',ylabel='mm',title='Índice climático no estacionario'); fig.tight_layout(); fig.savefig(out/'01_shift.png',dpi=140); plt.show()

In [ ]:
fig,ax=plt.subplots(); ax.hist(df.rain[:696],bins=35,density=True,alpha=.5,label='antes'); ax.hist(df.rain[696:],bins=35,density=True,alpha=.5,label='después'); ax.legend(); ax.set(xlabel='mm',ylabel='densidad',title='Distribución del índice por régimen'); fig.tight_layout(); fig.savefig(out/'02_distributions.png',dpi=140); plt.show()

## 5. Contrato parametrizado y controles

El trigger y agotamiento son parámetros contractuales versionados. La función debe ser monótona y acotada; otra clase de peligro puede invertir la dirección del índice.

In [ ]:
def payout(x,K,A,L):
 if A>=K or L<0: raise ValueError('require A<K and L>=0')
 return L*np.clip((K-np.asarray(x))/(K-A),0,1)
K,A,L=560.,400.,100.
df['pay']=payout(df.rain,K,A,L)
assert np.all(np.diff(payout(np.linspace(0,800,1000),K,A,L))<=1e-10)
assert payout(0,K,A,L)==L and payout(900,K,A,L)==0
print('contract invariants passed; activation rate =',np.mean(df.pay>0))

## 6. Cortes temporales; burn contra modelo condicional

Train/test respeta orden temporal. El baseline condicional ajusta una tendencia lineal y escala residual constante sobre train, simula la distribución futura de lluvia y calcula la esperanza del payout. Es deliberadamente sencillo: comparar contra GAM, regresión de cuantiles, GEV no estacionaria, pronósticos físicos y ensambles es trabajo futuro.

In [ ]:
n=len(df); cut=int(.70*n); tr=df.iloc[:cut]; te=df.iloc[cut:]
burn=float(tr.pay.mean()); beta=np.polyfit(tr.t,tr.rain,1); resid=tr.rain-np.polyval(beta,tr.t); sd=max(float(resid.std()),1.)
def cond_pay(mu,sd,K,A,L,seed=SEED,draws=2500):
 r=np.random.default_rng(seed); x=np.maximum(0,r.normal(np.asarray(mu)[:,None],sd,(len(mu),draws)))
 return payout(x,K,A,L).mean(axis=1)
mu=np.polyval(beta,te.t); pred=cond_pay(mu,sd,K,A,L); pure_cond=float(pred.mean()); realized=float(te.pay.mean())
pd.Series({'burn_train':burn,'conditional_forecast':pure_cond,'realized_test':realized})

## 7. Pesos exponenciales y pérdida de tamaño efectivo

Los pesos \(w_t\propto e^{-\gamma age_t}\) reaccionan a cambios recientes, pero pueden perseguir ruido. El tamaño efectivo \(n_{eff}=1/\sum \tilde w_t^2\) cuantifica cuánta historia informa la tarifa.

In [ ]:
rows=[]; z=tr.pay.to_numpy(); age=np.arange(len(z)-1,-1,-1)
for gamma in [0,.003,.01,.03]:
 w=np.exp(-gamma*age); w=w/w.sum(); rows.append({'gamma':gamma,'premium':w@z,'n_eff':1/(w@w)})
pd.DataFrame(rows)

## 8. Métricas de riesgo de base

Métricas reportadas: correlación pérdida-pago, shortfall medio, pago sin pérdida, reducción de varianza y pérdida severa sin activación. El umbral de pérdida severa se fija en train y se aplica al test.

In [ ]:
def basis(y,z,severe_cut):
 b=np.asarray(y)-np.asarray(z)
 return {'corr':float(np.corrcoef(y,z)[0,1]),'shortfall_mean':float(np.maximum(b,0).mean()),'overpay_mean':float(np.maximum(-b,0).mean()),'var_reduction':float(1-np.var(b)/(np.var(y)+1e-12)),'severe_loss_no_pay':float(np.mean((np.asarray(y)>=severe_cut)&(np.asarray(z)==0)))}
sev=float(tr.loss.quantile(.75)); testpay=payout(te.rain,K,A,L); basis(te.loss.to_numpy(),testpay,sev)

In [ ]:
fig,ax=plt.subplots(); ax.scatter(te.loss,testpay,s=12,alpha=.4); ax.set(xlabel='daño sintético',ylabel='pago',title='Pago paramétrico frente a pérdida económica'); fig.tight_layout(); fig.savefig(out/'03_basis.png',dpi=140); plt.show()

## 9. Selección de contrato dentro de train

Exploramos una grilla pequeña minimizando shortfall más penalización explícita al pago excedente. El test solo evalúa; nunca participa en selección. En práctica se incluirían restricciones de prima máxima, capital, aceptación y estabilidad por región.

In [ ]:
grid=[]
for kk in [500,540,580,620]:
 for aa in [300,360,420,460]:
  if aa>=kk: continue
  zz=payout(tr.rain,kk,aa,L); b=tr.loss.to_numpy()-zz
  grid.append((np.maximum(b,0).mean()+.25*np.maximum(-b,0).mean(),kk,aa))
score,kopt,aopt=min(grid); optpay=payout(te.rain,kopt,aopt,L)
print('train-selected K,A:',kopt,aopt,'train objective:',score,'test:',basis(te.loss.to_numpy(),optpay,sev))

## 10. Carga de cola y suficiencia

\(TVaR_\alpha=E[Z|Z\ge VaR_\alpha]\). Colas empíricas son inciertas con pocas catástrofes; la incertidumbre del cuantil debe acompañar la prima. Este cálculo no sustituye reaseguro ni capital regulatorio.

In [ ]:
def tvar(x,a):
 q=np.quantile(x,a); return float(np.asarray(x)[np.asarray(x)>=q].mean())
rows=[]
for alpha in [.90,.95,.975,.99]:
 tv=tvar(tr.pay,alpha)
 for lam in [0,.1,.25]: rows.append({'alpha':alpha,'lambda':lam,'pure':burn,'TVaR':tv,'loaded':burn+lam*max(tv-burn,0)})
pd.DataFrame(rows)

## 11. Asequibilidad y brecha explícita

La distribución de presupuestos también es sintética. No se reduce la prima técnica a una prima asequible: se calcula qué fracción podría pagarla y la brecha que tendría que financiarse con rediseño, adaptación, subsidio transparente o retención.

In [ ]:
bud=np.random.default_rng(SEED+8).lognormal(np.log(24),.45,5000); prices=np.linspace(5,70,100); uptake=np.array([(bud>=p).mean() for p in prices])
fig,ax=plt.subplots(); ax.plot(prices,uptake); ax.axvline(burn,ls='--',label='burn'); ax.axvline(pure_cond,ls=':',label='condicional'); ax.legend(); ax.set(xlabel='prima sintética',ylabel='fracción con presupuesto',ylim=(0,1),title='Asequibilidad ilustrativa'); fig.tight_layout(); fig.savefig(out/'04_affordability.png',dpi=140); plt.show()

## 12. Calibración por bloques fuera de muestra

Comparar primas emitidas en cada origen con pago promedio futuro en bloque. Una sola serie es evidencia descriptiva; replicar estaciones, zonas y años y usar bootstrap por bloques para intervalos.

In [ ]:
win=80; cal=[]
for start in range(cut,len(df)-win+1,win):
 hist=df.iloc[:start]; future=df.iloc[start:start+win]; bh=float(payout(hist.rain,K,A,L).mean())
 b=np.polyfit(hist.t,hist.rain,1); rr=hist.rain-np.polyval(b,hist.t); mm=np.polyval(b,future.t)
 cp=float(cond_pay(mm,max(rr.std(),1),K,A,L,seed=SEED+start,draws=1000).mean()); obs=float(payout(future.rain,K,A,L).mean())
 cal.append({'origin':start,'burn':bh,'conditional':cp,'realized':obs})
cal=pd.DataFrame(cal); cal

In [ ]:
fig,ax=plt.subplots(); ax.plot(cal.origin,cal.realized,'o-',label='pago OOS'); ax.plot(cal.origin,cal.burn,'s-',label='burn'); ax.plot(cal.origin,cal.conditional,'^-',label='condicional'); ax.legend(); ax.set(xlabel='origen',ylabel='pago promedio',title='Suficiencia por bloques temporales'); fig.tight_layout(); fig.savefig(out/'05_calibration.png',dpi=140); plt.show()

## 13. Contraejemplo estacionario y replicación

El método debe probarse en varios mundos, no una sola simulación. Medimos error absoluto de prima en réplicas estacionarias y con deriva. Métrica: \(|\hat\pi-E_{test}[Z]|\). Si el condicional no gana bajo deriva o pierde mucho bajo estacionariedad, ajustar la complejidad antes de proponerlo.

In [ ]:
rep=[]
for drift in [False,True]:
 for s in range(40):
  d=simulate(900,drift,s+100); a=d.iloc[:630]; b=d.iloc[630:]; bp=payout(a.rain,K,A,L).mean(); fit=np.polyfit(a.t,a.rain,1); e=a.rain-np.polyval(fit,a.t); cp=cond_pay(np.polyval(fit,b.t),max(e.std(),1),K,A,L,seed=s+300,draws=700).mean(); rp=payout(b.rain,K,A,L).mean()
  rep.append({'drift':drift,'replicate':s,'burn_error':abs(bp-rp),'conditional_error':abs(cp-rp)})
reps=pd.DataFrame(rep); reps.groupby('drift')[['burn_error','conditional_error']].agg(['mean','median'])

In [ ]:
fig,ax=plt.subplots()
for d,label in [(False,'stationary'),(True,'drift + break')]:
 s=reps[reps.drift==d]; ax.scatter(s.burn_error,s.conditional_error,label=label,alpha=.65)
mx=max(reps.burn_error.max(),reps.conditional_error.max()); ax.plot([0,mx],[0,mx],'k--'); ax.legend(); ax.set(xlabel='burn absolute error',ylabel='conditional absolute error',title='Each point is a synthetic replicate'); fig.tight_layout(); fig.savefig(out/'06_replications.png',dpi=140); plt.show()

## 14. Resultados, límites y falsación

No extrapolar el experimento sintético a tarifas, cultivos, Colombia, pérdidas, aseguradoras o proveedores reales. El resultado central a reportar es si el modelo condicional vence al burn por réplica y por régimen; el punto de corte, seed, varianza residual constante y distribución normal pueden favorecer o perjudicarlo. La carga de cola depende de frecuencia de excedencias y no corrige basis risk. Si el índice casi no informa daño, incluso una prima perfectamente suficiente para el pago puede dejar desprotegido al productor.

In [ ]:
result={'n':len(df),'train_n':len(tr),'test_n':len(te),'burn_train':float(burn),'conditional_forecast':float(pure_cond),'realized_test':float(realized),'selected_trigger_train':int(kopt),'selected_exhaustion_train':int(aopt),'replications':len(reps),'warning':'Synthetic experiment; no empirical climate or insurance inference.'}
pd.Series(result).to_json(out/'summary.json',indent=2); pd.Series(result)

## 15. Extensiones y ruta empírica

(1) GEV/GPD no estacionaria para colas; comparar niveles de retorno condicionales. (2) lluvia + humedad de suelo + evapotranspiración; selección por validación espacial. (3) optimizar trigger usando expectiles asimétricos y límites de solvencia. (4) ensemble climático con ambigüedad de pesos, sin inventar probabilidades. (5) pagos correlacionados entre zonas y acumulación de cartera. (6) adaptación observable y estimación causal del daño evitado. (7) riesgo conjunto de mora y liquidación del seguro. (8) sensor faltante, cambio de estación, revisiones, latencia y fallback. (9) control de volatilidad de tarifa sin ocultar costo. (10) uptake/asequibilidad con subsidio presupuestado.

## Fuentes académicas

Daron & Stainforth (2014), *Climate Risk Management*, DOI 10.1016/j.crm.2014.01.001; Ingels et al. (2024), *Annals of the NY Academy of Sciences*, DOI 10.1111/nyas.15255; Steinmann et al. (2023), *Environment Systems and Decisions*, DOI 10.1007/s10669-023-09934-x; Figueiredo et al. (2018), *Risk Analysis*; Zhang, Tan & Weng (2019), *ASTIN Bulletin*; Turvey et al. (2021), *European Review of Agricultural Economics*. La revisión de Ingels et al. resume la literatura de riesgo y seguro climático y destaca tarifas prospectivas, asequibilidad y brechas geográficas.

## Ejecución ilustrativa guardada

Con semilla fija, el benchmark burn de train produjo prima 7.29, el pronóstico condicional 30.15 y el pago promedio de test fue 40.79 miles COP sintéticos por exposición-periodo. El enfoque condicional subestimó el pago observado en esta trayectoria. El ajuste lineal de media y escala constante no resuelve el cambio abrupto de régimen.

Las métricas de base para el contrato de referencia fueron correlación 0.886, shortfall medio 6.61 y pago excedente medio 4.70 (miles COP sintéticos). La selección de trigger dentro de train eligió K=580, A=300. Las 40 réplicas por condición y seis figuras están en results/; consulte el README para los límites. Estos números solo verifican el mecanismo del DGP sintético.

![Cambio del índice](results/01_shift.png)

![Riesgo de base](results/03_basis.png)

![Asequibilidad](results/04_affordability.png)

## Reproducibilidad

Semilla fija 20261029. Ejecutar desde esta carpeta con las dependencias de `requirements.txt`. El notebook guarda seis figuras y `results/summary.json`. Validación local/Jupyter y evidencia empírica se mantienen separadas del código y del DGP sintético.